# 13 · Fail closed on schema drift and staleness

Test rejection paths against real source observations without overwriting a valid generation.

Open this notebook in Google Colab with a CPU runtime, or use a Linux x86-64
Jupyter environment with Python 3.11 or newer. No database account is required.
Run the cells in order. The first cell downloads a verified Kelvo preview binary
and installs pinned Python readers; it does not compile Kelvo.

Every analytical SQL query below runs through the **Kelvo executable**. PyArrow
reads its Arrow output, and pandas/matplotlib format small results. The pinned
DuckDB driver materializes execution before delivering Arrow batches; delivery
backpressure is not a worker memory bound.


**Data:** [Palmer Penguins](https://allisonhorst.github.io/palmerpenguins/),
[CC0 license](https://github.com/allisonhorst/palmerpenguins/blob/8957207b78d6ccd1b4654a9dd9c9041b657478ab/DESCRIPTION).
Citation: Horst, Hill and Gorman (2020), *palmerpenguins: Palmer Archipelago
(Antarctica) penguin data*, [DOI: 10.5281/zenodo.3960218](https://doi.org/10.5281/zenodo.3960218).
The helper downloads the pinned original CSV and verifies its SHA-256.


[All notebooks](https://github.com/SYNEHQ/kelvo-go/blob/cargo/notebooks/README.md)
| [Kelvo documentation](https://github.com/SYNEHQ/kelvo-go/blob/cargo/docs/README.md)

## Set up a private lab


In [ ]:
# Default setup installs pinned Python packages and verifies the reviewed helper.
# Operators validating a checkout can explicitly set KELVO_NOTEBOOK_SKIP_INSTALL=1
# and KELVO_NOTEBOOK_HELPER=/absolute/path/to/kelvo_notebooks.py before starting.
import hashlib
import importlib.util
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import urllib.parse
import urllib.request

KELVO_NOTEBOOK_REF = "eacb7e2aedd8020575ecfede53db819112c40199"
KELVO_HELPER_SHA256 = "86696a66a3c903eb8f18d529fcef055e8bee72234da324b8776529138436d9a9"
if os.environ.get("KELVO_NOTEBOOK_SKIP_INSTALL") != "1":
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--quiet",
        "pyarrow==25.0.1", "pandas==2.3.3", "matplotlib==3.10.7", "PyYAML==6.0.3",
    ], check=True)

helper_directory = None
local_helper = os.environ.get("KELVO_NOTEBOOK_HELPER")
if local_helper:
    helper_path = Path(local_helper).expanduser().resolve(strict=True)
else:
    helper_url = (
        "https://raw.githubusercontent.com/SYNEHQ/kelvo-go/"
        + KELVO_NOTEBOOK_REF + "/notebooks/kelvo_notebooks.py"
    )
    request = urllib.request.Request(helper_url, headers={"User-Agent": "kelvo-notebooks"})
    with urllib.request.urlopen(request, timeout=60) as response:
        final_url = response.geturl()
        if urllib.parse.urlsplit(final_url).scheme != "https" or final_url != helper_url:
            raise ValueError("Helper download did not retain its pinned HTTPS URL")
        payload = response.read(1024 * 1024 + 1)
    if len(payload) > 1024 * 1024:
        raise ValueError("Unexpected helper size")
    if hashlib.sha256(payload).hexdigest() != KELVO_HELPER_SHA256:
        raise ValueError("Helper checksum mismatch")
    helper_directory = Path(tempfile.mkdtemp(prefix="kelvo-notebook-helper-"))
    helper_path = helper_directory / "kelvo_notebooks.py"
    helper_path.write_bytes(payload)

spec = importlib.util.spec_from_file_location("kelvo_notebooks", helper_path)
helper = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = helper
spec.loader.exec_module(helper)
from kelvo_notebooks import Lab, remote_query

lab = Lab.create()
print("Kelvo workspace:", lab.root)


In [ ]:
import csv
import pyarrow as pa
import pyarrow.parquet as pq
import yaml
with lab.penguins().open(newline="") as stream:
    originals = list(csv.DictReader(stream))
typed = pa.table({
    "species": pa.array([row["species"] for row in originals], type=pa.string()),
    "body_mass_g": pa.array([None if row["body_mass_g"] in ("NA", "") else int(row["body_mass_g"])
                             for row in originals], type=pa.int32()),
})
source = lab.root / "typed-penguins.parquet"
pq.write_table(typed, source)
def snapshot_catalog(dataset, directory, max_age="1h"):
    return lab.catalog(
        sources=[{"id": "observations", "type": "parquet", "path": str(source)}],
        acceleration={"tenant_id": "notebook", "directory": str(directory), "datasets": [{
            "id": dataset,
            "query": {"mode": "federated", "sources": ["observations"], "sql": "SELECT * FROM observations"},
            "authorization_version": "public-data-v1", "refresh_interval": "0s", "max_age": max_age,
            "limits": {"max_rows": 10000, "max_bytes": 16777216,
                       "timeout": "60s", "memory_mb": 256, "threads": 2, "max_temp_mb": 256},
        }]},
    )
config = snapshot_catalog("stable", lab.root / "strict-snapshots")
first = yaml.safe_load(lab.command([
    "accelerate", "refresh", "--config", str(config), "--dataset", "stable"
]).stdout)
original_generation = first["snapshot"]["generation"]
baseline = lab.query(config, "SELECT * FROM stable ORDER BY species, body_mass_g NULLS FIRST",
                     sources=["stable"]).table()
assert baseline.num_rows == len(originals) == 344


## Reject an unapproved new column

The local source receives an extra nullable column. Strict schema matching is
the default; failure must preserve the prior generation and its exact result.


In [ ]:
changed = typed.append_column("unapproved_note", pa.array([None] * typed.num_rows, type=pa.string()))
pq.write_table(changed, source)
rejected = lab.command([
    "accelerate", "refresh", "--config", str(config), "--dataset", "stable"
], check=False)
assert rejected.returncode != 0
status = yaml.safe_load(lab.command([
    "accelerate", "status", "--config", str(config), "--dataset", "stable"
]).stdout)
assert status["snapshot"]["generation"] == original_generation
retained = lab.query(config, "SELECT * FROM stable ORDER BY species, body_mass_g NULLS FIRST",
                     sources=["stable"]).table()
assert retained.equals(baseline, check_metadata=True)
assert "unapproved_note" not in retained.column_names
pq.write_table(typed, source)
print("Schema drift was rejected; the complete prior snapshot remains readable.")


## Reject stale data

A separate demonstration catalog uses a one-nanosecond age budget so it becomes
stale without a long sleep. This is deliberately unsuitable for real service
configuration. Staleness must fail the query, not silently fall back to the
source or publish a partial export.


In [ ]:
stale_config = snapshot_catalog("stale", lab.root / "stale-snapshots", max_age="1ns")
lab.command(["accelerate", "refresh", "--config", str(stale_config), "--dataset", "stale"])
stale_status = yaml.safe_load(lab.command([
    "accelerate", "status", "--config", str(stale_config), "--dataset", "stale"
]).stdout)
assert stale_status["ready"] is False
output = lab.root / "stale-result.arrow"
stale_query = lab.command([
    "query", "--config", str(stale_config), "--sources", "stale",
    "--sql", "SELECT COUNT(*) FROM stale", "--out", str(output),
], check=False)
assert stale_query.returncode != 0
assert not output.exists()
print("Stale reads fail closed.")


For a deliberate schema rollout, review
[optional schema evolution](https://github.com/SYNEHQ/kelvo-go/blob/cargo/docs/schema-evolution.md).
Changing policy changes catalog identity and requires a complete successful
refresh. A restore does not renew timestamps or relax schema/authorization
checks. These notebook checks cover local behavior, not multi-node recovery.


## Clean up

Run this cell after inspecting or saving your results. Re-run the setup and data
cells if you want to restart the lesson after cleanup.


In [ ]:
# Close local processes and delete this lesson's private temporary workspace.
# Downloaded release/data caches, if configured by the helper, remain reusable.
lab.close()
if helper_directory is not None:
    shutil.rmtree(helper_directory, ignore_errors=True)
